In [1]:
import pandas as pd
import numpy as np
import GEOparse
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sklearn.svm import NuSVR
from sklearn.preprocessing import normalize
from scipy.optimize import nnls

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'
print("All imports successful ✅")

All imports successful ✅


In [2]:
print("Downloading GSE68465...")
print("~442 LUAD patients, has stage information")
print("May take 3-5 minutes...")

gse = GEOparse.get_GEO(geo="GSE68465",
                        destdir=f'{base}/data/external/',
                        silent=True)

print(f"\nDownload complete!")
print(f"Number of samples: {len(gse.gsms)}")
print(f"Platform: {list(gse.gpls.keys())}")

~442 LUAD patients, has stage information
May take 3-5 minutes...

Download complete!
Number of samples: 462
Platform: ['GPL96']


In [3]:
print("Extracting expression and clinical data...")

gsm_data     = {}
clinical_data = {}

for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and len(gsm.table) > 0:
        gsm_data[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']
    
    metadata = gsm.metadata
    clinical_data[gsm_name] = {
        'title': metadata.get('title', [''])[0],
        'characteristics': metadata.get('characteristics_ch1', [])
    }

print(f"Samples with expression: {len(gsm_data)}")

# Build expression matrix
expr_raw = pd.DataFrame(gsm_data).T
print(f"Raw expression matrix: {expr_raw.shape}")
print(f"First 5 probe IDs: {list(expr_raw.columns[:5])}")

# Extract clinical characteristics
survival_records = []
for gsm_name, data in clinical_data.items():
    record = {'sample_id': gsm_name}
    for c in data['characteristics']:
        if ':' in c:
            key, val = c.split(':', 1)
            record[key.strip()] = val.strip()
    survival_records.append(record)

survival_df = pd.DataFrame(survival_records).set_index('sample_id')
print(f"\nClinical columns: {list(survival_df.columns)}")
print(f"\nFirst 3 rows:")
print(survival_df.head(3))

Extracting expression and clinical data...
Samples with expression: 462
Raw expression matrix: (462, 22283)
First 5 probe IDs: ['AFFX-BioB-5_at', 'AFFX-BioB-M_at', 'AFFX-BioB-3_at', 'AFFX-BioC-5_at', 'AFFX-BioC-3_at']

Clinical columns: ['disease_state', 'Sex', 'age', 'race', 'vital_status', 'clinical_treatment_adjuvant_chemo', 'clinical_treatment_adjuvant_rt', 'disease_stage', 'first_progression_or_relapse', 'months_to_first_progression', 'mths_to_last_clinical_assessment', 'months_to_last_contact_or_death', 'smoking_history', 'surgical_margins', 'organism_part', 'histologic_grade']

First 3 rows:
                  disease_state     Sex age   race vital_status  \
sample_id                                                         
GSM1672281  Lung Adenocarcinoma  Female  74  White        Alive   
GSM1672282  Lung Adenocarcinoma  Female  74  White        Alive   
GSM1672283  Lung Adenocarcinoma  Female  74  White         Dead   

           clinical_treatment_adjuvant_chemo clinical_trea

In [4]:
# Check vital status and survival time
print("Vital status:")
print(survival_df['vital_status'].value_counts())
print(f"\nSurvival time sample:")
print(survival_df['months_to_last_contact_or_death'].head(5))
print(f"\nDisease stage values:")
print(survival_df['disease_stage'].value_counts().head(15))

Vital status:
vital_status
Dead     236
Alive    207
Name: count, dtype: int64

Survival time sample:
sample_id
GSM1672281    105.6
GSM1672282     25.2
GSM1672283     64.8
GSM1672284     69.2
GSM1672285     33.9
Name: months_to_last_contact_or_death, dtype: object

Disease stage values:
disease_stage
pN0pT2    162
pN0pT1    114
pN1pT2     55
pN2pT2     34
pN1pT1     24
pN0pT3     16
pN2pT1     11
pN0pT4      7
pN1pT3      7
pN2pT3      5
pN2pT4      3
pN1pT4      2
pp          2
pNXpT1      1
Name: count, dtype: int64


In [5]:
def parse_ptnm_stage(s):
    """Convert pTNM to stage grouping."""
    s = str(s).strip()
    try:
        # Extract N and T values
        n = int(s[2]) if 'N' in s and s[2].isdigit() else 0
        t = int(s[5]) if 'T' in s and s[5].isdigit() else 1
        
        # Stage grouping based on TNM
        if n == 0 and t in [1, 2]:
            return 'Stage I' if t == 1 else 'Stage II'
        elif n == 1 and t in [1, 2]:
            return 'Stage II'
        elif n == 2 or t in [3, 4]:
            return 'Stage III'
        else:
            return 'Stage I'
    except:
        return 'Unknown'

survival_df['stage_group'] = survival_df['disease_stage'].apply(parse_ptnm_stage)
print("Stage distribution:")
print(survival_df['stage_group'].value_counts())

# Build survival labels
# Convert months to days
survival_df['survival_days'] = pd.to_numeric(
    survival_df['months_to_last_contact_or_death'], errors='coerce') * 30.44

# Remove patients with missing survival
survival_df = survival_df[survival_df['survival_days'].notna()]
survival_df = survival_df[survival_df['vital_status'].isin(['Alive', 'Dead'])]

print(f"\nPatients with complete survival: {len(survival_df)}")
print(f"Events (deaths): {(survival_df['vital_status']=='Dead').sum()}")
print(f"Censored (alive): {(survival_df['vital_status']=='Alive').sum()}")
print(f"Survival range: {survival_df['survival_days'].min():.0f} to {survival_df['survival_days'].max():.0f} days")

# Build survival array
y_ext = np.array(
    [(vs == 'Dead', float(t)) for vs, t in 
     zip(survival_df['vital_status'], survival_df['survival_days'])],
    dtype=[('event', bool), ('time', float)]
)

# Clinical features
age_ext    = pd.to_numeric(survival_df['age'], errors='coerce').fillna(65)
gender_ext = (survival_df['Sex'] == 'Male').astype(float)

stage_II_ext  = (survival_df['stage_group'] == 'Stage II').astype(float)
stage_III_ext = (survival_df['stage_group'] == 'Stage III').astype(float)
stage_IV_ext  = (survival_df['stage_group'] == 'Stage IV').astype(float)

clinical_ext = pd.DataFrame({
    'age':             age_ext.values,
    'gender':          gender_ext.values,
    'stage_Stage II':  stage_II_ext.values,
    'stage_Stage III': stage_III_ext.values,
    'stage_Stage IV':  stage_IV_ext.values
}, index=survival_df.index)

print(f"\nClinical features shape: {clinical_ext.shape}")
print(f"Stage II:  {stage_II_ext.sum():.0f}")
print(f"Stage III: {stage_III_ext.sum():.0f}")
print(f"Stage IV:  {stage_IV_ext.sum():.0f}")

Stage distribution:
stage_group
Stage II     241
Stage I      136
Stage III     85
Name: count, dtype: int64

Patients with complete survival: 442
Events (deaths): 236
Censored (alive): 206
Survival range: 1 to 6210 days

Clinical features shape: (442, 5)
Stage II:  241
Stage III: 84
Stage IV:  0


In [6]:
# Get platform annotation
gpl = gse.gpls['GPL96']
print(f"Platform table shape: {gpl.table.shape}")
print(f"Platform columns: {list(gpl.table.columns)}")

# Build probe to gene symbol mapping
probe_to_gene = gpl.table.set_index('ID')['Gene Symbol'].dropna()
probe_to_gene = probe_to_gene[probe_to_gene != '']

print(f"Probes with gene symbols: {len(probe_to_gene)}")
print(f"Example: {dict(list(probe_to_gene.items())[:5])}")

# Filter expression to annotated probes
expr_filtered = expr_raw[[c for c in expr_raw.columns 
                           if c in probe_to_gene.index]]
expr_filtered.columns = [probe_to_gene[c] for c in expr_filtered.columns]

# Average duplicate gene symbols
expr_filtered = expr_filtered.astype(float)
expr_filtered = expr_filtered.T.groupby(level=0).mean().T

print(f"\nExpression after mapping: {expr_filtered.shape}")
print(f"Example genes: {list(expr_filtered.columns[:5])}")

# Align to patients with survival data
expr_filtered = expr_filtered.loc[
    expr_filtered.index.isin(survival_df.index)]
survival_df   = survival_df.loc[
    survival_df.index.isin(expr_filtered.index)]
clinical_ext  = clinical_ext.loc[
    clinical_ext.index.isin(expr_filtered.index)]

print(f"\nAligned patients: {len(expr_filtered)}")
print(f"Value range: {expr_filtered.values.min():.2f} to {expr_filtered.values.max():.2f}")
print(f"Log2 already? Max < 25: {expr_filtered.values.max() < 25}")

Platform table shape: (22283, 16)
Platform columns: ['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID', 'RefSeq Transcript ID', 'Gene Ontology Biological Process', 'Gene Ontology Cellular Component', 'Gene Ontology Molecular Function']
Probes with gene symbols: 21225
Example: {'1007_s_at': 'DDR1 /// MIR4640', '1053_at': 'RFC2', '117_at': 'HSPA6', '121_at': 'PAX8', '1255_g_at': 'GUCA1A'}

Expression after mapping: (462, 13515)
Example genes: ['A1CF', 'A2M', 'A4GALT', 'A4GNT', 'AAAS']

Aligned patients: 442
Value range: 0.06 to 130623.00
Log2 already? Max < 25: False


In [7]:
from datetime import datetime

# Log2(x+1) transform
expr_log2 = np.log2(expr_filtered.astype(float) + 1)
print(f"After log2 transform:")
print(f"  Min: {expr_log2.values.min():.2f}")
print(f"  Max: {expr_log2.values.max():.2f}")
print(f"  Mean: {expr_log2.values.mean():.2f}")

# Load LM22
lm22 = pd.read_csv(f'{base}/data/external/LM22.txt', sep='\t', index_col=0)

# Find common genes with LM22
common_lm22 = lm22.index.intersection(expr_log2.columns)
print(f"\nLM22 coverage: {len(common_lm22)}/{len(lm22.index)} ({len(common_lm22)/len(lm22.index)*100:.1f}%)")

expr_lm22  = expr_log2[common_lm22]
lm22_common = lm22.loc[common_lm22]

# DIY CIBERSORT
def run_cibersort_single(patient_expr, lm22_matrix):
    expr_linear = (2 ** patient_expr.values) - 1
    expr_linear = np.clip(expr_linear, 0, 1e6)
    lm22_linear = (2 ** lm22_matrix.values) - 1
    lm22_linear = np.clip(lm22_linear, 0, 1e6)
    lm22_norm   = normalize(lm22_linear, axis=0)
    expr_norm   = normalize(expr_linear.reshape(1, -1))[0]
    best_nu = 0.5; best_error = np.inf
    for nu in [0.25, 0.5, 0.75]:
        try:
            svr = NuSVR(nu=nu, kernel='linear', C=1.0)
            svr.fit(lm22_norm, expr_norm)
            error = np.mean((lm22_norm @ svr.coef_[0] - expr_norm) ** 2)
            if error < best_error:
                best_error = error; best_nu = nu
        except: continue
    try:
        svr = NuSVR(nu=best_nu, kernel='linear', C=1.0)
        svr.fit(lm22_norm, expr_norm)
        raw_weights = svr.coef_[0]
    except:
        raw_weights = np.zeros(lm22_matrix.shape[1])
    clipped = np.maximum(raw_weights, 0)
    if clipped.sum() == 0:
        clipped, _ = nnls(lm22_norm, expr_norm)
        clipped = np.maximum(clipped, 0)
    total = clipped.sum()
    final = clipped / total if total > 0 else np.ones(len(clipped)) / len(clipped)
    return dict(zip(lm22_matrix.columns, final))

print(f"\nRunning DIY CIBERSORT on {len(expr_lm22)} patients...")
print(f"Start: {datetime.now().strftime('%H:%M:%S')}")

results = {}
for i, pid in enumerate(expr_lm22.index):
    results[pid] = run_cibersort_single(expr_lm22.loc[pid], lm22_common)
    if (i+1) % 50 == 0 or i == 0:
        print(f"  {i+1}/{len(expr_lm22)} [{datetime.now().strftime('%H:%M:%S')}]")

immune_ext = pd.DataFrame(results).T
print(f"\nImmune features: {immune_ext.shape}")
print(f"Row sums: {immune_ext.sum(axis=1).mean():.4f}")
print(f"Macrophages M2 mean: {immune_ext['Macrophages M2'].mean():.4f}")

After log2 transform:
  Min: 0.09
  Max: 17.00
  Mean: 7.77

LM22 coverage: 517/547 (94.5%)

Running DIY CIBERSORT on 442 patients...
Start: 17:56:27
  1/442 [17:56:27]
  50/442 [17:56:28]
  100/442 [17:56:30]
  150/442 [17:56:32]
  200/442 [17:56:34]
  250/442 [17:56:35]
  300/442 [17:56:37]
  350/442 [17:56:39]
  400/442 [17:56:41]

Immune features: (442, 22)
Row sums: 1.0000
Macrophages M2 mean: 0.0579


In [8]:
from lifelines import CoxPHFitter

# Load GTEx reference
gtex_ref = json.load(open(f'{base}/data/processed/gtex_reference.json'))

# Compute dysregulation z-scores
dysreg_genes_available = [g for g in gtex_ref.keys() if g in expr_log2.columns]
print(f"GTEx reference genes available in GSE68465: {len(dysreg_genes_available)}")

dysreg_ext = pd.DataFrame(index=expr_log2.index)
for gene in dysreg_genes_available:
    gtex_mean = gtex_ref[gene]['mean']
    gtex_std  = gtex_ref[gene]['std']
    if gtex_std > 0:
        dysreg_ext[gene] = (expr_log2[gene] - gtex_mean) / gtex_std
    else:
        dysreg_ext[gene] = 0.0

print(f"Dysregulation matrix: {dysreg_ext.shape}")
print(f"Value range: {dysreg_ext.values.min():.2f} to {dysreg_ext.values.max():.2f}")

# Load training data
expr_train   = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg_train = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune_train = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical_tr  = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

common_tr    = expr_train.index.intersection(dysreg_train.index).intersection(
               immune_train.index).intersection(clinical_tr.index)
expr_train   = expr_train.loc[common_tr]
dysreg_train = dysreg_train.loc[common_tr]
immune_train = immune_train.loc[common_tr]
clinical_tr  = clinical_tr.loc[common_tr]

age_tr    = clinical_tr[['age']].copy()
gender_tr = (clinical_tr['gender'] == 'male').astype(float).to_frame()
stage_tr  = pd.get_dummies(clinical_tr['stage_group'], prefix='stage')
stage_tr  = stage_tr.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features_tr = pd.concat([age_tr, gender_tr, stage_tr],
                                   axis=1).astype(float).fillna(0)

y_train = np.array(
    [(bool(e), t) for e, t in zip(clinical_tr['event'], clinical_tr['survival_time'])],
    dtype=[('event', bool), ('time', float)])

# Get Lasso genes
cox_lasso = pickle.load(open(f'{base}/models/cox_lasso_expression.pkl', 'rb'))
gene_list = json.load(open(f'{base}/models/gene_list.json'))
coefs     = cox_lasso.coef_[:, 0]
lasso_genes = [g for g, s in zip(gene_list, coefs != 0) if s]

# Expression training with suffix
expr_tr_lasso = expr_train[lasso_genes].copy()
expr_tr_lasso.columns = [f"{g}_expr" for g in lasso_genes]

# Cox dysregulation selection on training
print("\nSelecting top 20 dysregulation genes...")
times  = y_train['time']
events = y_train['event']

cox_pvals_d = {}
for gene in dysreg_train.columns:
    try:
        df_tmp = pd.DataFrame({'T': times, 'E': events,
                               'gene': dysreg_train[gene].values})
        cph = CoxPHFitter()
        cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
        cox_pvals_d[gene] = cph.summary['p'].values[0]
    except:
        cox_pvals_d[gene] = 1.0
top_dysreg = list(pd.Series(cox_pvals_d).nsmallest(20).index)
print(f"Top dysreg genes: {len(top_dysreg)}")

dysreg_tr_sel = dysreg_train[top_dysreg].copy()
dysreg_tr_sel.columns = [f"{g}_dysreg" for g in top_dysreg]

# Interaction features training
interactions_tr = pd.DataFrame({
    'stageIII_x_M2':   (clinical_features_tr['stage_Stage III'] *
                        immune_train['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_features_tr['stage_Stage IV'] *
                        immune_train['T cells CD8']).values,
    'age_x_stageIII':  (clinical_features_tr['age'] *
                        clinical_features_tr['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_features_tr['stage_Stage III'] *
                        immune_train['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_train['Macrophages M2'] *
                        immune_train['T cells CD8']).values,
}, index=expr_train.index)

# Build training matrix
X_train = pd.concat([expr_tr_lasso, dysreg_tr_sel,
                      immune_train, clinical_features_tr,
                      interactions_tr], axis=1).fillna(0)

print(f"Training matrix: {X_train.shape}")

# Build external feature matrix
# Expression
expr_ext_lasso = pd.DataFrame(index=expr_log2.index)
for g in lasso_genes:
    if g in expr_log2.columns:
        expr_ext_lasso[f"{g}_expr"] = expr_log2[g].values
    else:
        expr_ext_lasso[f"{g}_expr"] = 0.0

# Dysregulation
dysreg_ext_sel = pd.DataFrame(index=dysreg_ext.index)
for g in top_dysreg:
    if g in dysreg_ext.columns:
        dysreg_ext_sel[f"{g}_dysreg"] = dysreg_ext[g].values
    else:
        dysreg_ext_sel[f"{g}_dysreg"] = 0.0

# Interaction features external
interactions_ext = pd.DataFrame({
    'stageIII_x_M2':   (clinical_ext['stage_Stage III'] *
                        immune_ext['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_ext['stage_Stage IV'] *
                        immune_ext['T cells CD8']).values,
    'age_x_stageIII':  (clinical_ext['age'] *
                        clinical_ext['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_ext['stage_Stage III'] *
                        immune_ext['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_ext['Macrophages M2'] *
                        immune_ext['T cells CD8']).values,
}, index=immune_ext.index)

# Build external matrix
X_ext = pd.concat([expr_ext_lasso, dysreg_ext_sel,
                    immune_ext, clinical_ext,
                    interactions_ext], axis=1).fillna(0)

# Reorder to match training
X_ext = X_ext[X_train.columns]

print(f"External matrix: {X_ext.shape}")
print(f"Columns match:   {list(X_ext.columns) == list(X_train.columns)}")
print(f"Any NaN:         {X_ext.isna().any().any()}")

# Scale
scaler    = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train),
                           columns=X_train.columns, index=X_train.index)
X_ext_s   = pd.DataFrame(scaler.transform(X_ext),
                           columns=X_ext.columns, index=X_ext.index)

# Train and predict
model = GradientBoostingSurvivalAnalysis(
    n_estimators=300, learning_rate=0.05, max_depth=2,
    min_samples_split=20, min_samples_leaf=10,
    subsample=0.8, random_state=42)
model.fit(X_train_s, y_train)

risk_scores = model.predict(X_ext_s)

# Rebuild y_ext aligned
y_ext = np.array(
    [(vs == 'Dead', float(t)) for vs, t in
     zip(survival_df.loc[X_ext.index]['vital_status'],
         survival_df.loc[X_ext.index]['survival_days'])],
    dtype=[('event', bool), ('time', float)])

ci_ext = concordance_index_censored(
    y_ext['event'].astype(bool),
    y_ext['time'],
    risk_scores)[0]

print(f"\n{'='*45}")
print(f"EXTERNAL VALIDATION RESULT (GSE68465)")
print(f"{'='*45}")
print(f"Cohort:          GSE68465 (LUAD, microarray)")
print(f"Patients:        {len(y_ext)}")
print(f"Events (deaths): {y_ext['event'].sum()} ({y_ext['event'].mean()*100:.1f}%)")
print(f"C-index:         {ci_ext:.3f}")
print(f"{'='*45}")
print(f"\nFull external validation summary:")
print(f"  TCGA training: 0.702")
print(f"  GSE72094:      0.636")
print(f"  CPTAC-LUAD:    0.557 (no stage, imputed censoring)")
print(f"  GSE68465:      {ci_ext:.3f}")

GTEx reference genes available in GSE68465: 497
Dysregulation matrix: (442, 497)
Value range: -2.72 to 91.90

Selecting top 20 dysregulation genes...
Top dysreg genes: 20
Training matrix: (478, 124)
External matrix: (442, 124)
Columns match:   True
Any NaN:         False

EXTERNAL VALIDATION RESULT (GSE68465)
Cohort:          GSE68465 (LUAD, microarray)
Patients:        442
Events (deaths): 236 (53.4%)
C-index:         0.637

Full external validation summary:
  TCGA training: 0.702
  GSE72094:      0.636
  CPTAC-LUAD:    0.557 (no stage, imputed censoring)
  GSE68465:      0.637


In [9]:
import os
os.makedirs(f'{base}/outputs/results', exist_ok=True)

results_all = {
    "training": {
        "cohort": "TCGA-LUAD",
        "n_patients": 478,
        "n_events": 121,
        "c_index": 0.702,
        "method": "leakage-free 5-fold StratifiedKFold"
    },
    "external_1": {
        "cohort": "GSE72094",
        "n_patients": 398,
        "n_events": 113,
        "c_index": 0.636,
        "platform": "Affymetrix microarray",
        "notes": "no dysregulation (platform mismatch with GTEx)"
    },
    "external_2": {
        "cohort": "GSE68465",
        "n_patients": 442,
        "n_events": 236,
        "c_index": 0.637,
        "platform": "Affymetrix microarray GPL96",
        "notes": "full features including dysregulation"
    },
    "external_3": {
        "cohort": "CPTAC-LUAD",
        "n_patients": 203,
        "n_events": 53,
        "c_index": 0.557,
        "platform": "RNA-seq",
        "notes": "no stage information, alive patients censored at max death time"
    }
}

with open(f'{base}/outputs/results/all_validation_results.json', 'w') as f:
    json.dump(results_all, f, indent=2)

print("Saved: outputs/results/all_validation_results.json")
print(f"\nKey finding: GSE72094 (0.636) and GSE68465 (0.637)")
print(f"Two independent cohorts confirming generalisation ✅")

Saved: outputs/results/all_validation_results.json

Key finding: GSE72094 (0.636) and GSE68465 (0.637)
Two independent cohorts confirming generalisation ✅
